In [1]:
import imaplib
import email
import re
from email.header import decode_header

In [2]:
# Configuration

IMAP_SERVER  = "imap.gmail.com"   # Gmail | Outlook: "outlook.office365.com"
EMAIL_USER   = "fodhq.as.fbd@gmail.com"
EMAIL_PASS   = "xcas ciae zgba ldto"  # Use an App Password, NOT your real password
MAILBOX      = "INBOX"
KEYWORD_LIST = ['KACHCHH','BANASKANTHA','PATAN','MEHSANA','MAHESANA','SABARKANTHA',
                'GANDHINAGAR','AHMEDABAD','SURENDRANAGAR','RAJKOT','JAMNAGAR',
                'PORBANDAR','JUNAGADH','JUNAGARH','AMRELI','AMRELLI','BHAVNAGAR',
                'ANAND','KHEDA','PANCHMAHAL','DOHAD','DAHOD','VADODARA','BARODA',
                'NARMADA','BHARUCH','SURAT','DANGS','DANG','NAVSARI','VALSAD','TAPI',
                'ARVALLI','ARAVALLI','BOTAD','CHHOTA UDEPUR','CHHOTA UDAIPUR',
                'DEVBHUMI DWARKA','GIR SOMNATH','MAHISAGAR','MORBI']  # ← your words here

In [3]:
def decode_str(value):
    """Decode encoded email header strings (e.g., =?UTF-8?...?)."""
    if value is None:
        return ""
    parts = decode_header(value)
    decoded = []
    for part, charset in parts:
        if isinstance(part, bytes):
            decoded.append(part.decode(charset or "utf-8", errors="replace"))
        else:
            decoded.append(part)
    return " ".join(decoded)

In [4]:
def extract_email_address(from_header):
    """Pull bare email address from a From header like 'Name <email@x.com>'."""
    match = re.search(r"<(.+?)>", from_header)
    if match:
        return match.group(1).strip().lower()
    return from_header.strip().lower()

In [5]:
def get_email_body(msg):
    """Extract plain-text body from a (possibly multipart) email message."""
    body = ""
    if msg.is_multipart():
        for part in msg.walk():
            if part.get_content_type() == "text/plain":
                charset = part.get_content_charset() or "utf-8"
                try:
                    body += part.get_payload(decode=True).decode(charset, errors="replace")
                except Exception:
                    pass
    else:
        charset = msg.get_content_charset() or "utf-8"
        try:
            body = msg.get_payload(decode=True).decode(charset, errors="replace")
        except Exception:
            pass
    return body

In [6]:
def search_by_keyword(imap, keyword):
    """
    Search mailbox for emails containing a keyword in SUBJECT or BODY.
    Returns a set of message IDs.
    """
    msg_ids = set()

    # Search in SUBJECT
    status, data = imap.search(None, f'SUBJECT "{keyword}"')
    if status == "OK" and data[0]:
        msg_ids.update(data[0].split())

    # Search in BODY (full text)
    status, data = imap.search(None, f'BODY "{keyword}"')
    if status == "OK" and data[0]:
        msg_ids.update(data[0].split())

    return msg_ids

In [7]:
def fetch_senders(keyword_list):
    """
    Connect to the mail server, search for emails containing any of the
    given keywords, and return a dict mapping each keyword to sender emails.
    """
    senders_by_keyword = {kw: set() for kw in keyword_list}
    all_matching_ids   = set()
    keyword_to_ids     = {}

    with imaplib.IMAP4_SSL(IMAP_SERVER) as imap:
        imap.login(EMAIL_USER, EMAIL_PASS)
        imap.select(MAILBOX, readonly=True)

        print(f"[✓] Connected to {IMAP_SERVER} — Mailbox: {MAILBOX}\n")

        # 1. Gather all matching message IDs per keyword
        for keyword in keyword_list:
            ids = search_by_keyword(imap, keyword)
            keyword_to_ids[keyword] = ids
            all_matching_ids.update(ids)
            print(f"  Keyword '{keyword}': {len(ids)} email(s) found")

        print(f"\n[→] Total unique emails to process: {len(all_matching_ids)}\n")

        # 2. Fetch each unique message once and extract sender
        id_to_sender = {}
        for msg_id in all_matching_ids:
            status, data = imap.fetch(msg_id, "(RFC822.HEADER)")
            if status != "OK":
                continue
            msg = email.message_from_bytes(data[0][1])
            raw_from = decode_str(msg.get("From", ""))
            sender   = extract_email_address(raw_from)
            id_to_sender[msg_id] = sender

        # 3. Map keyword → set of sender addresses
        for keyword, ids in keyword_to_ids.items():
            for msg_id in ids:
                if msg_id in id_to_sender:
                    senders_by_keyword[keyword].add(id_to_sender[msg_id])

    return senders_by_keyword

In [8]:
def main():
    results = fetch_senders(KEYWORD_LIST)

    print("=" * 55)
    print("  SENDER EMAIL ADDRESSES BY KEYWORD")
    print("=" * 55)

    all_senders = set()
    for keyword, senders in results.items():
        print(f"\n📌 Keyword: '{keyword}' ({len(senders)} unique sender(s))")
        for sender in sorted(senders):
            print(f"    → {sender}")
        all_senders.update(senders)

    print("\n" + "=" * 55)
    print(f"  TOTAL UNIQUE SENDERS ACROSS ALL KEYWORDS: {len(all_senders)}")
    print("=" * 55)
    for sender in sorted(all_senders):
        print(f"  {sender}")

    # Optional: save to a text file
    with open("senders.txt", "w") as f:
        f.write("\n".join(sorted(all_senders)))
    print("\n[✓] Results saved to senders.txt")

In [9]:
if __name__ == "__main__":
    main()

[✓] Connected to imap.gmail.com — Mailbox: INBOX

  Keyword 'KACHCHH': 8 email(s) found
  Keyword 'BANASKANTHA': 20 email(s) found
  Keyword 'PATAN': 13 email(s) found
  Keyword 'MEHSANA': 28 email(s) found
  Keyword 'MAHESANA': 25 email(s) found
  Keyword 'SABARKANTHA': 11 email(s) found
  Keyword 'GANDHINAGAR': 125 email(s) found
  Keyword 'AHMEDABAD': 482 email(s) found
  Keyword 'SURENDRANAGAR': 24 email(s) found
  Keyword 'RAJKOT': 27 email(s) found
  Keyword 'JAMNAGAR': 25 email(s) found
  Keyword 'PORBANDAR': 7 email(s) found
  Keyword 'JUNAGADH': 5 email(s) found
  Keyword 'JUNAGARH': 2 email(s) found
  Keyword 'AMRELI': 15 email(s) found
  Keyword 'AMRELLI': 0 email(s) found
  Keyword 'BHAVNAGAR': 28 email(s) found
  Keyword 'ANAND': 68 email(s) found
  Keyword 'KHEDA': 19 email(s) found
  Keyword 'PANCHMAHAL': 14 email(s) found
  Keyword 'DOHAD': 1 email(s) found
  Keyword 'DAHOD': 9 email(s) found
  Keyword 'VADODARA': 160 email(s) found
  Keyword 'BARODA': 540 email(s) foun